# AGNO Personal Finance Agent

This notebook demonstrates how to use the NeMo Agent Toolkit SDK to create a personal finance planning workflow using AGNO agents.

## Key Features

1. **AGNO Framework Integration** - Uses AGNO agents for research and planning
2. **Environment Variable Handling** - Tests how API keys from env vars are serialized
3. **Multi-Agent Workflow** - Researcher + Planner agents

## Prerequisites

- Install the packages:
  ```bash
  pip install -e examples/frameworks/agno_personal_finance
  pip install nvidia-nat-agno
  ```
- Set environment variables:
  - `OPENAI_API_KEY` - OpenAI API key
  - `SERP_API_KEY` - SerpAPI key for web search


In [ ]:
import os
import sys

# Add src to path for development
module_path = os.path.abspath('../../../src/')
if module_path not in sys.path:
    sys.path.insert(0, module_path)


In [ ]:
import logging

logger = logging.getLogger()
logger.setLevel(logging.INFO)


## Environment Variable Handling Test

This section tests whether environment variable references are preserved when saving configs.
The expected behavior is that `${OPENAI_API_KEY}` should remain as-is in the saved YAML,
not be replaced with the actual API key value.


In [ ]:
# Check if environment variables are set
env_vars = {
    "OPENAI_API_KEY": os.environ.get("OPENAI_API_KEY", "NOT SET"),
    "SERP_API_KEY": os.environ.get("SERP_API_KEY", "NOT SET"),
}

for name, value in env_vars.items():
    if value == "NOT SET":
        print(f"⚠️  {name} is not set")
    else:
        # Only show first 8 chars for security
        masked = value[:8] + "..." if len(value) > 8 else value
        print(f"✓ {name} is set: {masked}")


## Creating the Workflow

We'll create the AGNO Personal Finance workflow using the SDK classes.


In [ ]:
from pathlib import Path

from nat.llm.openai_llm import OpenAILLM

# Import the SerpAPI tool
from nat.plugins.agno.tools.serp_api_tool import SerpApiTool
from nat.utils.sdk.nat_env_var import NatEnvironmentVariable
from nat.utils.sdk.nat_workflow import NatWorkflow

# Import the AGNO personal finance function
from nat_agno_personal_finance.agno_personal_finance_function import AgnoPersonalFinanceFunction

# Create the OpenAI LLM with API key from environment variable
# Using NatEnvironmentVariable preserves the ${VAR_NAME} reference when saving
openai_llm = OpenAILLM(
    model_name="gpt-4o",
    temperature=0.0,
    # Use api_key_env to preserve the env var reference in saved config
    api_key_env=NatEnvironmentVariable("OPENAI_API_KEY"),
    name="openai_llm",
)

# Create the SerpAPI search tool
# Using NatEnvironmentVariable preserves the ${VAR_NAME} reference when saving
web_search_tool = SerpApiTool(
    api_key_env=NatEnvironmentVariable("SERP_API_KEY"),
    max_results=5,
    name="web_search_tool",
)

# Create the personal finance function
# The tool_objects parameter accepts NatFunction objects and converts them to FunctionRefs
personal_finance = AgnoPersonalFinanceFunction(
    llm=openai_llm,
    tool_objects=[web_search_tool],
    name="personal_finance",
)

# Wrap in NatWorkflow
nat_workflow = NatWorkflow(
    entrypoint=personal_finance,
)

print("Workflow created successfully!")


## Testing Environment Variable Serialization

This is the critical test - when we save the config, we want to verify if the API keys are:
1. ❌ Serialized as actual values (security risk, not portable)
2. ✅ Preserved as environment variable references like `${OPENAI_API_KEY}`


In [ ]:
# Save the workflow configuration
config_dir = Path(os.getcwd()) / "config"
config_dir.mkdir(parents=True, exist_ok=True)

config_path = config_dir / "workflow_config.yaml"
nat_workflow.save_to_config_file(config_path)

print(f"Configuration saved to: {config_path}")
print("\n" + "="*60)
print("SAVED CONFIGURATION:")
print("="*60 + "\n")

with open(config_path) as f:
    config_content = f.read()
    print(config_content)


In [ ]:
# Analyze the saved config for API key handling
import re

print("="*60)
print("API KEY SERIALIZATION ANALYSIS:")
print("="*60 + "\n")

# Check if actual API key values were serialized (security issue)
openai_key = os.environ.get("OPENAI_API_KEY", "")
serp_key = os.environ.get("SERP_API_KEY", "")

issues = []

if openai_key and openai_key in config_content:
    issues.append("❌ OPENAI_API_KEY was serialized as its actual value!")
elif "${OPENAI_API_KEY}" in config_content:
    print("✅ OPENAI_API_KEY preserved as env var reference: ${OPENAI_API_KEY}")
elif "api_key:" in config_content:
    # Check what's after api_key
    api_key_match = re.search(r'api_key:\s*(.+)', config_content)
    if api_key_match:
        value = api_key_match.group(1).strip()
        if value and value != "null":
            issues.append(f"⚠️  api_key serialized as: {value[:20]}...")
        else:
            print("ℹ️  api_key is null (will load from env at runtime)")

if serp_key and serp_key in config_content:
    issues.append("❌ SERP_API_KEY was serialized as its actual value!")
elif "${SERP_API_KEY}" in config_content:
    print("✅ SERP_API_KEY preserved as env var reference: ${SERP_API_KEY}")

if issues:
    print("\n⚠️  ISSUES FOUND:")
    for issue in issues:
        print(f"  {issue}")
    print("\nThis is a potential security risk - API keys should not be saved to config files!")
else:
    print("\n✅ No obvious API key leakage detected")


## Compare with Original YAML Config

Let's compare the SDK-generated config with the original YAML to see how environment variables are handled.


In [ ]:
# Read the original YAML config for comparison
original_config_path = Path(os.getcwd()).parent / "configs" / "config.yml"

print("="*60)
print("ORIGINAL YAML CONFIGURATION:")
print("="*60 + "\n")

with open(original_config_path) as f:
    original_content = f.read()
    print(original_content)

print("\n" + "="*60)
print("KEY DIFFERENCES:")
print("="*60)
print("""
The original config uses environment variable references like:
  - api_key: ${OPENAI_API_KEY}
  - api_key: ${SERP_API_KEY}

When we create components via the SDK and pass actual values from os.environ,
these get serialized as the actual values, not as env var references.

This is a known limitation - the SDK doesn't track the *source* of a value,
only the value itself.
""")


## Summary

This notebook demonstrated:

1. **AGNO Framework Integration** - Using AGNO agents (Researcher + Planner) for personal finance
2. **SDK Component Creation** - Creating LLMs, tools, and functions via the SDK
3. **Environment Variable Handling** - Using `NatEnvironmentVariable` to preserve env var refs

### Using NatEnvironmentVariable

The `NatEnvironmentVariable` class allows you to specify environment variable references
that are preserved when saving configs.

**Usage:**
```python
from nat.utils.sdk.nat_env_var import NatEnvironmentVariable

# Use api_key_env instead of api_key
llm = OpenAILLM(
    api_key_env=NatEnvironmentVariable("OPENAI_API_KEY"),
    model_name="gpt-4o",
)

tool = SerpApiTool(
    api_key_env=NatEnvironmentVariable("SERP_API_KEY"),
)
```

**Result in saved config:**
```yaml
llms:
  openai_llm:
    _type: openai
    api_key: ${OPENAI_API_KEY}  # Preserved as env var reference!
    model: gpt-4o
```

This approach:
- ✅ Resolves env vars at runtime for actual use
- ✅ Preserves `${VAR_NAME}` format when saving configs
- ✅ Makes configs portable and secure (no secrets in files)
